In [1]:
import os
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', '')
os.environ['GEMINI_API_KEY'] = os.getenv('GOOGLE_API_KEY', '')

In [ ]:
import os
from agent.agent_svg import Agent
from render_svg import SVGAgent

In [ ]:
model_name = "o3"
task_id = 'task_svg_2'
# tgt_img_path = f"/Users/zhanghantao/Desktop/agent-synth/{task_id}.png"
tgt_img_path = "/home/jingyang/agent-synth/data/tinysvg/manual_scene.png"
agent = Agent(model_name=model_name, target_image_path=tgt_img_path)
agent_png = SVGAgent(600, 600)

In [ ]:
init_expression = agent.initialize()
print(init_expression)

In [ ]:
output_path = f'test_agent_svg/{task_id}'
os.makedirs(output_path, exist_ok=True)

In [ ]:
# init_expression = [{'shape_type': 'ellipse', 'x': 270, 'y': 260, 'scale_x': 20, 'scale_y': 40, 'fill_color': 'red', 'stroke_color': 'none'}, {'shape_type': 'rectangle', 'x': 330, 'y': 230, 'scale_x': 30, 'scale_y': 120, 'fill_color': 'blue', 'stroke_color': 'none'}, {'shape_type': 'ellipse', 'x': 270, 'y': 310, 'scale_x': 80, 'scale_y': 25, 'fill_color': 'green', 'stroke_color': 'none'}, {'shape_type': 'rectangle', 'x': 330, 'y': 300, 'scale_x': 40, 'scale_y': 40, 'fill_color': 'purple', 'stroke_color': 'none'}]

In [ ]:
agent_png.create_from_dict(init_expression)
agent_png.save_png(os.path.join(output_path, "initial.png"))

In [ ]:
new_expression, step_info, improvement_made = agent.optimization_step(current_image_path=f'/home/jingyang/agent-synth/test_agent_svg/{task_id}/initial.png', current_expression=init_expression, output_path=output_path)
print(new_expression)
print(step_info)

In [ ]:
agent_png.clear()
agent_png.create_from_dict(new_expression)
agent_png.save_png(os.path.join(output_path, "optimized_1.png"))

In [ ]:
i = 1
while i < 5:
    new_expression, step_info, improvement_made = agent.optimization_step(current_image_path=f'/home/jingyang/agent-synth/test_agent_svg/{task_id}/optimized_{i}.png', current_expression=new_expression, output_path=output_path)
    print(f"Step {i+1}: Improvement made - ", improvement_made)
    print(step_info)
    if improvement_made:
        agent_png.clear()
        agent_png.create_from_dict(new_expression)
        i += 1
        agent_png.save_png(os.path.join(output_path, f"optimized_{i}.png"))

In [1]:
# test the vlm judgment
from agent.agent_svg import Agent
import os
from render_svg import SVGAgent
model_name = "o3"
    
img_dir = "/home/jingyang/agent-synth/data/render_svg_mutator"
img_paths = sorted(os.listdir(img_dir))

# FIX: Create full paths by joining with the directory
tgt_img_path = os.path.join(img_dir, img_paths[0])
cndt_img_paths = [os.path.join(img_dir, path) for path in img_paths[1:]]

agent = Agent(model_name=model_name, target_image_path=tgt_img_path)
agent_png = SVGAgent(600, 600)

res = agent.run_tournament(target_image_path=tgt_img_path,
                           candidate_paths=cndt_img_paths)

print("Tournament result:", res)

Tournament result: {'champion': '/home/jingyang/agent-synth/data/render_svg_mutator/fine_control_0.02.png', 'tournament_log': [{'round': 1, 'matches': [{'match_number': 1, 'candidate_1': '/home/jingyang/agent-synth/data/render_svg_mutator/fine_control_0.02.png', 'candidate_2': '/home/jingyang/agent-synth/data/render_svg_mutator/fine_control_0.15.png', 'winner': '/home/jingyang/agent-synth/data/render_svg_mutator/fine_control_0.02.png', 'scores': {'candidate_1': 9.3, 'candidate_2': 8.5}, 'confidence': 0.7, 'score_difference': 0.8000000000000007, 'vlm_analysis': 'GENERAL DESCRIPTION  \nTarget – White canvas with four objects:  \n• Purple circle (black stroke) in the upper-left quadrant.  \n• Orange diamond (square rotated 45°, thin blue stroke) in upper-right.  \n• Pink tilted ellipse (red stroke) in lower-left.  \n• Green axis-aligned square (black stroke) in lower-right.  \nAll strokes look ~6 px except the diamond (~4 px). Objects are of comparable visual size and well separated.\n\n-

In [3]:
# save the result to a txt file
import json
output_path = f'test_agent_svg/tournament_result.txt'
with open(output_path, 'w') as f:
    # res is a dictionary, convert it to a string
    res = json.dumps(res, indent=4)
    f.write(res)

In [ ]:
# # plot the current and target images
# import matplotlib.pyplot as plt
# import cv2 as cv
# curr_img = cv.imread(current_image_path)
# targ_img = cv.imread(target_image_path)

# plt.figure(figsize=(10, 5))
# plt.subplot(1, 2, 1)
# plt.imshow(cv.cvtColor(curr_img, cv.COLOR_BGR2RGB))
# plt.title("Current Image")
# plt.axis('off')

# plt.subplot(1, 2, 2)
# plt.imshow(cv.cvtColor(targ_img, cv.COLOR_BGR2RGB))
# plt.title("Target Image")
# plt.axis('off')

# plt.show()

# """
# Response: VISUAL DISCREPANCIES 
# 1. A large red-outlined ellipse has appeared to the right of the blue oval in the CURRENT image – this shape does not exist in the TARGET. 
# 2. The blue vertical rectangle that should sit just to the right of the blue oval is completely missing. 
# 3. The purple square is: • too wide (it became a small rectangle), • has an unwanted 2 px black stroke, • and is mis-positioned ±20 px left of where it should be (it now sits under the blue oval instead of under the missing blue rectangle). 
# 4. The tiny green dot is roughly correct, but sits ±5 px too low (it should be vertically centred between the blue oval and purple square). 
#     GEOMETRIC ISSUES 
#         • Blue oval geometry is fine; the red ellipse is an unwarranted duplicate that is ~140 % larger. 
#         • Purple square lost its proportions (needs to be a perfect square, not a wider rectangle). 
#         • Vertical alignment: in the TARGET all three column elements (oval / rectangle / square) line up on the same x-column; the CURRENT breaks that column. 
#     STYLING PROBLEMS 
#         • Strokes: only flat fills are required – no strokes at all. Remove black stroke on the purple square and red stroke/fill on the rogue ellipse. 
#         • Purple square should use the same fill as TARGET (#800080); blue rectangle/oval use #0000FF; green dot uses #00C800. 
#         • Stroke-width, line-caps, dashes, opacity – none of these should be set; keep everything default/none. LAYOUT & COMPOSITION Approximate correct anchor positions (relative to 600×600 canvas centre): 
#         • Blue oval: centre ≈ (295, 310) 
#         • Blue vertical rectangle (8 × 20): top-left ≈ (315, 300) • Purple square (8 × 8): top-left ≈ (315, 325) 
#         • Green dot (4 × 4): centre ≈ (300, 320) 
        

# OPTIMIZATION RECOMMENDATIONS (CODE-LEVEL) 
# 1. Delete the entire <ellipse> element that has fill/stroke red. 
# 2. Add a new element for the missing vertical rectangle: <rect x="315" y="300" width="8" height="20" fill="#0000FF"/> 
# 3. Reduce the existing purple rectangle to an 8 × 8 square, move it under the new blue rectangle, and strip its stroke: <rect x="315" y="325" width="8" height="8" fill="#800080"/> 
# 4. Remove stroke attributes globally unless explicitly needed: search/replace stroke="..." or stroke-width="..." on every element and delete for these shapes. 
# 5. Nudge the green dot upward ~5 px to vertically centre it: <rect x="298" y="318" width="4" height="4" fill="#00C800"/> 
# 6. For filesize: collapse redundant whitespace, merge style attributes into a short ‘style’ string, and group identical fills: e.g. <g fill="#0000FF">… 
# 7. Export without unnecessary metadata/comment blocks to shrink the SVG. 

# PRIORITY FIXES (highest impact) 
# 1. REMOVE the extra red ellipse and the black stroke around the purple square. 
# 2. ADD the missing blue vertical rectangle and reposition the purple square directly beneath it. 
# 3. Correct sizing: purple element must be an 8 px square; green dot must sit 5 px higher.

# SEMANTIC UNDERSTANDING The graphic appears to be an abstract “three-dot” or “traffic-light” style icon composed of small colored primitives. The CURRENT version breaks this minimalist intent by introducing an unrelated red element and heavy strokes. Fixing the items above restores the intended clean, simple icon.
# """